# Brazil DI Curve — BQNT Download and Audit

This notebook records the BQuant/BQNT workflow used during the jury-revision audit of the nominal Brazilian DI reference curve for 2010–2025.

It is intended to be stored next to:

- `README.md`
- `di_curve.v6.xlsx`
- `di_futures_2010_2025_raw.xlsx` (generated here)
- `di_generic_mapping_od1_od60_2010_2025.xlsx` (generated here)

## What the notebook reproduces

1. The 186 canonical monthly observation dates.
2. Historical Bloomberg generic-to-specific ODA mapping.
3. Exact-date futures `PX_LAST`.
4. Actual `FUTURES_VALUATION_DATE`.
5. Month-to-date futures volume reconstructed from daily `PX_VOLUME`.
6. Independent OD1–OD60 historical mapping audit.
7. Historical ODA-chain audit using `bq.univ.futures("ODA Comdty", dates=...)`.
8. Final structural checks against `di_curve.v6.xlsx` if that workbook is placed in the notebook working directory.

## Important scope note

Cash (`BZDIOVRA Index`) and pre × DI swap quotes were refreshed in Bloomberg Excel with exact-date daily `BDH`, then pasted as values into the final workbook. The exact formula is documented near the end of this notebook and in `README.md`. The BQNT code below records the futures download and mapping workflow actually used during the audit.

In [ ]:
import bql
import pandas as pd

from pathlib import Path
from urllib.parse import quote
from IPython.display import HTML, display

bq = bql.Service()

print("BQL service initialized.")

## BQuant browser-download helper

This helper saves output in the active BQuant/Jupyter project and creates a browser download button.

In [ ]:
def download_file(file_path, obj=None, index=False):
    path = Path(file_path)

    if not path.is_absolute():
        path = Path.cwd() / path

    path.parent.mkdir(parents=True, exist_ok=True)

    if obj is not None:
        ext = path.suffix.lower()

        if isinstance(obj, (pd.DataFrame, pd.Series)):
            if ext == ".xlsx":
                obj.to_excel(path, index=index)
            elif ext == ".csv":
                obj.to_csv(path, index=index)
            else:
                raise ValueError("For pandas objects use .xlsx or .csv")

        elif isinstance(obj, str):
            path.write_text(obj, encoding="utf-8")

        elif isinstance(obj, bytes):
            path.write_bytes(obj)

        else:
            raise TypeError(f"Unsupported object type: {type(obj)}")

    if not path.exists():
        raise FileNotFoundError(path)

    cwd = Path.cwd()
    relative_path = path.relative_to(cwd)

    encoded_path = "/".join(
        quote(part)
        for part in relative_path.parts
    )

    filename = path.name

    html = f"""
    <button
        style="padding: 8px 16px; font-size: 14px; cursor: pointer;"
        onclick="
            const currentPath = window.location.pathname;
            const basePath = currentPath.split('/lab/')[0];
            const downloadUrl =
                window.location.origin
                + basePath
                + '/files/{encoded_path}?download=1';

            const a = document.createElement('a');
            a.href = downloadUrl;
            a.download = '{filename}';
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        "
    >
        ⬇ Download {filename}
    </button>
    """

    print("Saved:")
    print(path)
    display(HTML(html))

    return path

## 1. Canonical monthly date grid

The final sample contains 186 months from January 2010 through June 2025. Twenty-one initial month-end dates were replaced with the preceding usable trading date established during the data audit.

In [ ]:
base_dates = pd.date_range(
    start="2010-01-01",
    end="2025-06-30",
    freq="BM",
)

replacement_map = {
    pd.Timestamp("2010-12-31"): pd.Timestamp("2010-12-30"),
    pd.Timestamp("2011-12-30"): pd.Timestamp("2011-12-29"),
    pd.Timestamp("2012-12-31"): pd.Timestamp("2012-12-28"),
    pd.Timestamp("2013-03-29"): pd.Timestamp("2013-03-28"),
    pd.Timestamp("2013-12-31"): pd.Timestamp("2013-12-30"),
    pd.Timestamp("2014-12-31"): pd.Timestamp("2014-12-30"),
    pd.Timestamp("2015-12-31"): pd.Timestamp("2015-12-30"),
    pd.Timestamp("2016-12-30"): pd.Timestamp("2016-12-29"),
    pd.Timestamp("2017-02-28"): pd.Timestamp("2017-02-24"),
    pd.Timestamp("2017-12-29"): pd.Timestamp("2017-12-28"),
    pd.Timestamp("2018-03-30"): pd.Timestamp("2018-03-29"),
    pd.Timestamp("2018-05-31"): pd.Timestamp("2018-05-30"),
    pd.Timestamp("2018-12-31"): pd.Timestamp("2018-12-28"),
    pd.Timestamp("2019-12-31"): pd.Timestamp("2019-12-30"),
    pd.Timestamp("2020-12-31"): pd.Timestamp("2020-12-30"),
    pd.Timestamp("2021-12-31"): pd.Timestamp("2021-12-30"),
    pd.Timestamp("2022-02-28"): pd.Timestamp("2022-02-25"),
    pd.Timestamp("2022-12-30"): pd.Timestamp("2022-12-29"),
    pd.Timestamp("2023-12-29"): pd.Timestamp("2023-12-28"),
    pd.Timestamp("2024-03-29"): pd.Timestamp("2024-03-28"),
    pd.Timestamp("2024-12-31"): pd.Timestamp("2024-12-30"),
}

canonical_dates = [
    replacement_map.get(pd.Timestamp(d), pd.Timestamp(d))
    for d in base_dates
]

assert len(canonical_dates) == 186
assert len(set(canonical_dates)) == 186

print("Canonical dates:", len(canonical_dates))
print("First date:", canonical_dates[0].date())
print("Last date:", canonical_dates[-1].date())
print("Replacement dates:", len(replacement_map))

## 2. Historical ODA chain — independent audit

This function queries the actual ODA futures chain as it existed on the historical date. It was used to verify that market prices and actual maturities were available independently of any manually constructed generic labels.

Daily volume is accumulated from the first calendar day of the month through the canonical observation date so that no later-date information is used.

In [ ]:
def fetch_oda_chain_full(d):
    """Historical ODA futures chain with exact price, maturity, and MTD volume."""

    chain_date = pd.Timestamp(d).normalize()
    bql_date = chain_date.strftime("%Y-%m-%d")
    month_start = chain_date.replace(day=1).strftime("%Y-%m-%d")

    universe = bq.univ.futures(
        "ODA Comdty",
        dates=bql_date,
    )

    price_items = {
        "Ticker": bq.data.id(),
        "PX_LAST": bq.data.px_last(dates=bql_date),
        "FUTURES_VALUATION_DATE": bq.data.futures_valuation_date(),
    }

    price_resp = bq.execute(
        bql.Request(universe, price_items)
    )

    out = pd.concat(
        [x.df() for x in price_resp],
        axis=1,
    )

    out = out[
        [
            "Ticker",
            "PX_LAST",
            "FUTURES_VALUATION_DATE",
        ]
    ].copy()

    out["Ticker"] = out["Ticker"].astype(str).str.strip()
    out["PX_LAST"] = pd.to_numeric(out["PX_LAST"], errors="coerce")
    out["FUTURES_VALUATION_DATE"] = pd.to_datetime(
        out["FUTURES_VALUATION_DATE"],
        errors="coerce",
    )

    volume_items = {
        "VOLUME": bq.data.px_volume(
            dates=bq.func.range(
                month_start,
                bql_date,
            )
        )
    }

    volume_resp = bq.execute(
        bql.Request(universe, volume_items)
    )

    vol = volume_resp[0].df().reset_index()
    vol["ID"] = vol["ID"].astype(str).str.strip()
    vol["VOLUME"] = pd.to_numeric(vol["VOLUME"], errors="coerce")

    volume_mtd = (
        vol
        .groupby("ID", as_index=False)["VOLUME"]
        .sum(min_count=1)
    )

    volume_map = volume_mtd.set_index("ID")["VOLUME"]
    out["VOLUME"] = out["Ticker"].map(volume_map)

    out = (
        out
        .sort_values("FUTURES_VALUATION_DATE", na_position="last")
        .reset_index(drop=True)
    )

    out["CHAIN_POSITION"] = range(1, len(out) + 1)
    out["CHAIN_DATE"] = chain_date

    return out[
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "Ticker",
            "FUTURES_VALUATION_DATE",
            "PX_LAST",
            "VOLUME",
        ]
    ]

# Example:
# display(fetch_oda_chain_full("2024-12-30").head(15))

## 3. Futures acquisition used for the `v6` audit file

`di_curve.v6.xlsx` preserves the earlier 46 generic futures slots:

`OD1–OD43, OD54, OD57, OD59`.

For each canonical date, Bloomberg's historical `FUT_CUR_GEN_TICKER` is used to resolve each generic ticker to a dated ODA contract. The dated contract then supplies the exact-date price, actual futures valuation date, and reconstructed MTD volume.

This section reproduces `di_futures_2010_2025_raw.xlsx`.

In [ ]:
V6_GENERIC_NUMBERS = list(range(1, 44)) + [54, 57, 59]
V6_GENERIC_TICKERS = [f"OD{i} Comdty" for i in V6_GENERIC_NUMBERS]

assert len(V6_GENERIC_TICKERS) == 46


def normalize_specific_ticker(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if not x:
        return pd.NA

    if x.lower().endswith("comdty"):
        return x

    return f"{x} Comdty"


def clean_response_item(response_item, value_column):
    df = response_item.df().reset_index()

    if "ID" not in df.columns:
        df = df.rename(columns={df.columns[0]: "ID"})

    if value_column not in df.columns:
        raise ValueError(
            f"Column '{value_column}' not found. "
            f"Returned columns: {list(df.columns)}"
        )

    df = df[["ID", value_column]].copy()
    df["ID"] = df["ID"].astype(str).str.strip()

    return df

In [ ]:
def fetch_di_futures_for_date(d):
    """Reconstruct the v6 generic futures block for one canonical date."""

    curve_date = pd.Timestamp(d).normalize()
    bql_date = curve_date.strftime("%Y-%m-%d")
    month_start = curve_date.replace(day=1).strftime("%Y-%m-%d")

    # A. Historical generic -> specific contract mapping
    mapping_items = {
        "SPECIFIC_TICKER": bq.data.fut_cur_gen_ticker(
            dates=bql_date
        )
    }

    mapping_resp = bq.execute(
        bql.Request(
            V6_GENERIC_TICKERS,
            mapping_items,
        )
    )

    mapping = clean_response_item(
        mapping_resp[0],
        "SPECIFIC_TICKER",
    )

    mapping = mapping.rename(
        columns={"ID": "Bloomberg generic ticker"}
    )

    mapping["Generic number"] = (
        mapping["Bloomberg generic ticker"]
        .str.extract(r"OD(\d+)", expand=False)
        .astype("Int64")
    )

    mapping["Generic ticker"] = mapping["Bloomberg generic ticker"]

    mapping["Specific ticker"] = (
        mapping["SPECIFIC_TICKER"]
        .apply(normalize_specific_ticker)
    )

    specific_tickers = (
        mapping["Specific ticker"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    # B. Exact-date prices
    if specific_tickers:
        price_items = {
            "PX_LAST": bq.data.px_last(dates=bql_date)
        }

        price_resp = bq.execute(
            bql.Request(specific_tickers, price_items)
        )

        prices = clean_response_item(price_resp[0], "PX_LAST")
        prices = prices.rename(columns={"ID": "Specific ticker"})
        prices["PX_LAST"] = pd.to_numeric(
            prices["PX_LAST"],
            errors="coerce",
        )
    else:
        prices = pd.DataFrame(columns=["Specific ticker", "PX_LAST"])

    # C. Actual futures valuation/maturity date
    if specific_tickers:
        maturity_items = {
            "FUTURES_VALUATION_DATE": bq.data.futures_valuation_date()
        }

        maturity_resp = bq.execute(
            bql.Request(specific_tickers, maturity_items)
        )

        maturity = clean_response_item(
            maturity_resp[0],
            "FUTURES_VALUATION_DATE",
        )

        maturity = maturity.rename(columns={"ID": "Specific ticker"})
        maturity["FUTURES_VALUATION_DATE"] = pd.to_datetime(
            maturity["FUTURES_VALUATION_DATE"],
            errors="coerce",
        )
    else:
        maturity = pd.DataFrame(
            columns=["Specific ticker", "FUTURES_VALUATION_DATE"]
        )

    # D. Month-to-date daily PX_VOLUME accumulation
    if specific_tickers:
        volume_items = {
            "VOLUME": bq.data.px_volume(
                dates=bq.func.range(
                    month_start,
                    bql_date,
                )
            )
        }

        volume_resp = bq.execute(
            bql.Request(specific_tickers, volume_items)
        )

        volume_daily = volume_resp[0].df().reset_index()

        if "ID" not in volume_daily.columns:
            volume_daily = volume_daily.rename(
                columns={volume_daily.columns[0]: "ID"}
            )

        volume_daily["ID"] = volume_daily["ID"].astype(str).str.strip()
        volume_daily["VOLUME"] = pd.to_numeric(
            volume_daily["VOLUME"],
            errors="coerce",
        )

        volume_mtd = (
            volume_daily
            .groupby("ID", as_index=False)["VOLUME"]
            .sum(min_count=1)
            .rename(
                columns={
                    "ID": "Specific ticker",
                    "VOLUME": "volume",
                }
            )
        )
    else:
        volume_mtd = pd.DataFrame(
            columns=["Specific ticker", "volume"]
        )

    out = mapping.merge(
        prices,
        on="Specific ticker",
        how="left",
    )

    out = out.merge(
        maturity,
        on="Specific ticker",
        how="left",
    )

    out = out.merge(
        volume_mtd,
        on="Specific ticker",
        how="left",
    )

    out["CHAIN_DATE"] = curve_date
    out["Type"] = "future"
    out = out.rename(columns={"PX_LAST": "px_last"})

    order_map = {
        n: i
        for i, n in enumerate(V6_GENERIC_NUMBERS)
    }

    out["_order"] = out["Generic number"].map(order_map)

    out = (
        out
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    return out[
        [
            "CHAIN_DATE",
            "Generic number",
            "Generic ticker",
            "Bloomberg generic ticker",
            "Specific ticker",
            "FUTURES_VALUATION_DATE",
            "px_last",
            "volume",
            "Type",
        ]
    ]

### Run the full 186-date futures download

This cell can make many Bloomberg requests. Run it only in an authorized BQuant session.

In [ ]:
frames = []
errors = []

for i, d in enumerate(canonical_dates, start=1):
    print(
        f"{i:03d}/{len(canonical_dates)}  "
        f"{d.strftime('%Y-%m-%d')}"
    )

    try:
        frames.append(
            fetch_di_futures_for_date(d)
        )
    except Exception as exc:
        print("  ERROR:", exc)
        errors.append(
            {
                "CHAIN_DATE": d,
                "ERROR": str(exc),
            }
        )

if not frames:
    raise RuntimeError("No DI futures data were downloaded.")

di_futures_raw = pd.concat(
    frames,
    ignore_index=True,
)

errors_df = pd.DataFrame(errors)

print("\nDOWNLOAD SUMMARY")
print("Canonical dates:", len(canonical_dates))
print("Dates downloaded:", di_futures_raw["CHAIN_DATE"].nunique())
print("Generic futures per date:", len(V6_GENERIC_TICKERS))
print("Expected rows:", 186 * 46)
print("Actual rows:", len(di_futures_raw))
print("Non-missing PX_LAST:", di_futures_raw["px_last"].notna().sum())
print("Non-missing volume:", di_futures_raw["volume"].notna().sum())
print("Errors:", len(errors_df))

In [ ]:
# Known anchor checks used during the audit.
anchor_check = di_futures_raw[
    (
        (di_futures_raw["CHAIN_DATE"] == pd.Timestamp("2010-12-30"))
        & (di_futures_raw["Generic ticker"] == "OD1 Comdty")
    )
    |
    (
        (di_futures_raw["CHAIN_DATE"] == pd.Timestamp("2024-12-30"))
        & (di_futures_raw["Generic ticker"] == "OD1 Comdty")
    )
]

display(
    anchor_check[
        [
            "CHAIN_DATE",
            "Generic ticker",
            "Specific ticker",
            "FUTURES_VALUATION_DATE",
            "px_last",
            "volume",
        ]
    ]
)

# Expected approximate audit values:
# 2010-12-30 | OD1 | ODF11 Comdty | 10.650 | 15,290,095
# 2024-12-30 | OD1 | ODF25 Comdty | 12.154 | 11,617,795

In [ ]:
futures_output = "di_futures_2010_2025_raw.xlsx"

with pd.ExcelWriter(
    futures_output,
    engine="openpyxl",
) as writer:
    di_futures_raw.to_excel(
        writer,
        sheet_name="DI Futures Raw",
        index=False,
    )

    pd.DataFrame(
        {"CHAIN_DATE": canonical_dates}
    ).to_excel(
        writer,
        sheet_name="Canonical Dates",
        index=False,
    )

    errors_df.to_excel(
        writer,
        sheet_name="Errors",
        index=False,
    )

download_file(futures_output)

## 4. OD1–OD60 historical generic-mapping audit

The purpose of this audit is to remove reliance on a manually constructed `od + month number` label. Bloomberg itself is queried for the generic-to-specific relationship on each historical date.

The audit performed during the revision found:

- OD1–OD48 mapped to a specific underlying contract on every canonical date.
- OD49–OD60 returned no underlying contract on any canonical date.
- No specific contract mapped to more than one generic ticker on the same date.

The code below reproduces that check.

In [ ]:
AUDIT_GENERIC_TICKERS = [
    f"OD{i} Comdty"
    for i in range(1, 61)
]


def fetch_generic_mapping(d):
    chain_date = pd.Timestamp(d).normalize()
    bql_date = chain_date.strftime("%Y-%m-%d")

    items = {
        "UNDERLYING_TICKER": bq.data.fut_cur_gen_ticker(
            dates=bql_date
        )
    }

    resp = bq.execute(
        bql.Request(
            AUDIT_GENERIC_TICKERS,
            items,
        )
    )

    mapping = pd.concat(
        [x.df() for x in resp],
        axis=1,
    ).reset_index()

    if "ID" in mapping.columns:
        mapping = mapping.rename(columns={"ID": "Generic ticker"})
    else:
        mapping = mapping.rename(
            columns={mapping.columns[0]: "Generic ticker"}
        )

    if "DATE" in mapping.columns:
        mapping = mapping.drop(columns=["DATE"])

    mapping["Generic ticker"] = (
        mapping["Generic ticker"]
        .astype(str)
        .str.strip()
    )

    mapping["Ticker"] = (
        mapping["UNDERLYING_TICKER"]
        .astype("string")
        .str.strip()
    )

    mapping["Ticker full"] = mapping["Ticker"].apply(
        normalize_specific_ticker
    )

    mapping["Generic number"] = (
        mapping["Generic ticker"]
        .str.extract(r"OD(\d+)", expand=False)
        .astype("Int64")
    )

    mapping["CHAIN_DATE"] = chain_date

    return mapping[
        [
            "CHAIN_DATE",
            "Generic number",
            "Generic ticker",
            "Ticker",
            "Ticker full",
        ]
    ]

In [ ]:
mapping_frames = []
mapping_errors = []

for i, d in enumerate(canonical_dates, start=1):
    print(
        f"{i:03d}/{len(canonical_dates)}  "
        f"{d.strftime('%Y-%m-%d')}"
    )

    try:
        mapping_frames.append(
            fetch_generic_mapping(d)
        )
    except Exception as exc:
        print("  ERROR:", exc)
        mapping_errors.append(
            {
                "CHAIN_DATE": d,
                "ERROR": str(exc),
            }
        )

generic_mapping = pd.concat(
    mapping_frames,
    ignore_index=True,
)

mapping_errors_df = pd.DataFrame(mapping_errors)

mapped_only = generic_mapping[
    generic_mapping["Ticker full"].notna()
].copy()

duplicate_specific = (
    mapped_only
    .groupby(
        ["CHAIN_DATE", "Ticker full"],
        as_index=False,
    )
    .agg(
        generic_count=("Generic ticker", "nunique"),
        generic_tickers=(
            "Generic ticker",
            lambda x: ", ".join(sorted(set(x))),
        ),
    )
)

duplicate_specific = duplicate_specific[
    duplicate_specific["generic_count"] > 1
].copy()

availability = (
    generic_mapping
    .groupby("Generic number")["Ticker full"]
    .count()
)

print("\nGENERIC MAPPING SUMMARY")
print("Dates:", generic_mapping["CHAIN_DATE"].nunique())
print("Generics per date:", 60)
print("Expected rows:", 186 * 60)
print("Actual rows:", len(generic_mapping))
print("Non-missing mappings:", generic_mapping["Ticker"].notna().sum())
print("Query errors:", len(mapping_errors_df))
print(
    "Date + ticker combinations mapping to >1 generic:",
    len(duplicate_specific),
)

print("\nMapping availability by generic number:")
print(availability.to_string())

In [ ]:
mapping_output = "di_generic_mapping_od1_od60_2010_2025.xlsx"

with pd.ExcelWriter(
    mapping_output,
    engine="openpyxl",
) as writer:
    generic_mapping.to_excel(
        writer,
        sheet_name="Generic Mapping",
        index=False,
    )

    duplicate_specific.to_excel(
        writer,
        sheet_name="Duplicate Check",
        index=False,
    )

    mapping_errors_df.to_excel(
        writer,
        sheet_name="Errors",
        index=False,
    )

download_file(mapping_output)

## 5. Cross-check raw futures against the independent OD1–OD60 mapping

This is the key audit join:

```text
CHAIN_DATE
+
raw Specific ticker
=
OD1–OD60 mapping Ticker full
```

For every raw futures row with a non-null specific ticker, the independent mapping should return the same Bloomberg generic ticker.

In [ ]:
raw_nonnull = di_futures_raw[
    di_futures_raw["Specific ticker"].notna()
].copy()

mapping_nonnull = generic_mapping[
    generic_mapping["Ticker full"].notna()
].copy()

cross = raw_nonnull.merge(
    mapping_nonnull[
        [
            "CHAIN_DATE",
            "Generic ticker",
            "Ticker full",
        ]
    ].rename(
        columns={
            "Generic ticker": "Audit generic ticker",
        }
    ),
    left_on=["CHAIN_DATE", "Specific ticker"],
    right_on=["CHAIN_DATE", "Ticker full"],
    how="left",
    validate="many_to_one",
)

cross["GENERIC_MATCH"] = (
    cross["Bloomberg generic ticker"]
    == cross["Audit generic ticker"]
)

print("Raw rows with non-null specific ticker:", len(raw_nonnull))
print("Rows matched to independent mapping:", cross["Audit generic ticker"].notna().sum())
print("Generic-label matches:", cross["GENERIC_MATCH"].sum())
print("Generic-label mismatches:", (~cross["GENERIC_MATCH"]).sum())

if (~cross["GENERIC_MATCH"]).any():
    display(
        cross.loc[
            ~cross["GENERIC_MATCH"],
            [
                "CHAIN_DATE",
                "Specific ticker",
                "Bloomberg generic ticker",
                "Audit generic ticker",
            ],
        ].head(50)
    )

## 6. Cash and swap exact-date Bloomberg refresh

The cash and swap legs were refreshed in Bloomberg Excel rather than BQNT during this workflow.

The exact-date daily formula used was:

```excel
=BDH($F2,L$1,$E2,$E2,"per=d","days=trading","CDR=BZ","dts=h","QtTyp=P")
```

This replaced the earlier monthly-history formula because `per=m` produced zeros on some corrected observation dates even though exact daily observations existed.

Final workbook conventions:

- Cash `BZDIOVRA Index`: `Term = 1/252` on every date.
- Cash `volume = 1,000,000` as a schema-compatibility sentinel.
- Swap `volume = 1,000,000` as a schema-compatibility sentinel.
- These sentinel values are **not market trading volumes**.
- Missing swap prices remain missing; they are not backfilled from other dates.

## 7. Optional validation of `di_curve.v6.xlsx`

Place `di_curve.v6.xlsx` in the notebook working directory and run this cell. It verifies the structural properties documented in `README.md`.

Note: the current `v6` snapshot intentionally still contains the legacy 46 futures slots. The OD1–OD60 audit above is retained so the final curve-construction step can explicitly decide whether to move to OD1–OD48.

In [ ]:
final_path = Path("di_curve.v6.xlsx")

if not final_path.exists():
    print(
        "di_curve.v6.xlsx not found in the current directory. "
        "Copy it here to run the final audit."
    )
else:
    final = pd.read_excel(
        final_path,
        sheet_name="only_values",
    )

    final["Curve date"] = pd.to_datetime(
        final["Curve date"],
        errors="coerce",
    )

    final["type_norm"] = (
        final["Type"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    final["px_num"] = pd.to_numeric(
        final["px_last"],
        errors="coerce",
    )

    final["vol_num"] = pd.to_numeric(
        final["volume"],
        errors="coerce",
    )

    final["term_num"] = pd.to_numeric(
        final["Term"],
        errors="coerce",
    )

    print("FINAL V6 AUDIT")
    print("Rows:", len(final))
    print("Dates:", final["Curve date"].nunique())
    print("First date:", final["Curve date"].min())
    print("Last date:", final["Curve date"].max())
    print("Rows/date:", final.groupby("Curve date").size().value_counts().to_dict())
    print("Types:", final["type_norm"].value_counts().to_dict())

    cash = final[final["type_norm"] == "cash"]
    futures = final[final["type_norm"] == "future"]
    swaps = final[final["type_norm"] == "swap"]

    assert len(final) == 13_392
    assert final["Curve date"].nunique() == 186
    assert (final.groupby("Curve date").size() == 72).all()

    assert len(cash) == 186
    assert len(futures) == 8_556
    assert len(swaps) == 4_650

    assert cash["term_num"].eq(1 / 252).all()
    assert cash["vol_num"].eq(1_000_000).all()
    assert swaps["vol_num"].eq(1_000_000).all()

    print("Cash prices present:", cash["px_num"].notna().sum(), "/", len(cash))
    print("Swap prices present:", swaps["px_num"].notna().sum(), "/", len(swaps))
    print(
        "Futures rows with specific ticker:",
        futures["Specific ticker"].notna().sum(),
        "/",
        len(futures),
    )

    replacement_dates = set(replacement_map.values())
    present_dates = set(final["Curve date"].dropna())

    print(
        "Corrected dates present:",
        len(replacement_dates & present_dates),
        "/",
        len(replacement_dates),
    )

    print("\nPASS: documented v6 structural checks completed.")

## 8. Downstream curve-construction rules

The workbook is a market-data staging layer. The actual curve builder should:

1. work **within the same observation date** only;
2. never average curve observations across dates;
3. apply the `volume > 1000` screen to **futures only**;
4. treat the cash/swap `1,000,000` value as a sentinel, not an observed volume;
5. remove missing/zero price nodes before interpolation;
6. preserve overlapping raw observations but apply an explicit source-priority/splice rule before interpolation;
7. use the actual futures maturity field where appropriate rather than inferring tenor from a manually generated generic number;
8. keep enough identifiers to trace each selected curve node back to its Bloomberg observation.

See `README.md` for the full methodological history and known OD44–OD48 review point.